In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
# Load data
with open('lidl_clean.json') as f:
    data = json.load(f)

# Build receipts dataframe
receipts = pd.DataFrame([{
    'date': r['date'],
    'store': r['store'],
    'total': r['total'],
    'num_items': len(r['items'])
} for r in data])

receipts['date'] = pd.to_datetime(receipts['date'])
receipts['month'] = receipts['date'].dt.to_period('M')

print(f"Receipts: {len(receipts)}")
print(f"Date range: {receipts['date'].min().date()} to {receipts['date'].max().date()}")
print(f"Total spent: €{receipts['total'].sum():.2f}")
print(f"Avg basket: €{receipts['total'].mean():.2f}")
print(f"\nTop stores:")
print(receipts['store'].value_counts().head())

In [ ]:
items_df = pd.DataFrame([
    {
        'date': r['date'],
        'store': r['store'],
        'name': i['name'],
        'quantity': i['quantity'],
        'unit_price': i['unit_price'],
        'total_price': i['total_price'],
        'tax_type': i['tax_type'],
    }
    for r in data for i in r['items']
])

items_df['date'] = pd.to_datetime(items_df['date'])
items_df['month'] = items_df['date'].dt.to_period('M')

print(f"Total rows: {len(items_df)}")
print(f"Columns: {list(items_df.columns)}")
items_df.head(20)


In [ ]:
# Get top 10 most purchased items
top_10_names = (
    items_df.groupby('name')['quantity']
    .count()
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)

# Filter to only those items
top_items_monthly = items_df[items_df['name'].isin(top_10_names)].copy()

# Group by month and item name
monthly_items = (
    top_items_monthly
    .groupby(['month', 'name'])['total_price']
    .sum()
    .reset_index()
    .sort_values('month')
)

monthly_items['month_str'] = monthly_items['month'].astype(str)
months_ordered = sorted(monthly_items['month_str'].unique().tolist())

# Plot
fig, ax = plt.subplots(figsize=(15, 6))

for name in top_10_names:
    subset = monthly_items[monthly_items['name'] == name].sort_values('month')
    ax.plot(subset['month_str'], subset['total_price'], marker='o', label=name)

ax.set_title('Monthly Spend on Top 10 Most Purchased Items', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('€ Spent')
ax.set_xticks(months_ordered)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Table sorted by month asc
pivot = (
    monthly_items
    .pivot(index='month_str', columns='name', values='total_price')
    .fillna(0)
    .round(2)
    .loc[months_ordered]  # force correct row order
)
pivot

In [ ]:
monthly_spend = (
    receipts.groupby('month')['total']
    .sum()
    .reset_index()
    .sort_values('month')
)
monthly_spend['month_str'] = monthly_spend['month'].astype(str)

# Add 3-month rolling average
monthly_spend['rolling_avg'] = monthly_spend['total'].rolling(3, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(monthly_spend['month_str'], monthly_spend['total'], color='#0066cc', alpha=0.6, label='Monthly Spend')
ax.plot(monthly_spend['month_str'], monthly_spend['rolling_avg'], color='#e8472a', linewidth=2, marker='o', label='3-Month Avg')
ax.set_title('Monthly Total Spend at Lidl', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('€ Spent')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(monthly_spend[['month_str', 'total', 'rolling_avg']].to_string(index=False))

In [ ]:
# Find items purchased in both early 2024 and 2025/2026
items_df['year'] = items_df['date'].dt.year

early = items_df[items_df['month'].astype(str) <= '2024-08'].groupby('name')['unit_price'].mean()
late  = items_df[items_df['month'].astype(str) >= '2025-06'].groupby('name')['unit_price'].mean()

inflation = pd.DataFrame({'price_2024': early, 'price_2025_26': late}).dropna()
inflation['change_pct'] = ((inflation['price_2025_26'] - inflation['price_2024']) / inflation['price_2024'] * 100).round(1)
inflation = inflation[inflation['price_2024'] > 0].sort_values('change_pct', ascending=False)

print("=== Price Increases ===")
print(inflation.head(10).to_string())
print("\n=== Price Decreases ===")
print(inflation.tail(10).to_string())

# Plot top 15 changes
top_changes = pd.concat([inflation.head(8), inflation.tail(7)]).sort_values('change_pct')
colors = ['#0066cc' if x < 0 else '#e8472a' for x in top_changes['change_pct']]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_changes.index, top_changes['change_pct'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Price Changes: 2024 vs 2025/26', fontsize=14)
ax.set_xlabel('% Change')
plt.tight_layout()
plt.show()

In [ ]:
# Total months in dataset
total_months = monthly_spend['month_str'].nunique()

# Count how many distinct months each item appears in
loyalty = (
    items_df.groupby('name')['month']
    .nunique()
    .reset_index()
    .rename(columns={'month': 'months_purchased'})
)
loyalty['loyalty_score'] = (loyalty['months_purchased'] / total_months * 100).round(1)
loyalty = loyalty.sort_values('loyalty_score', ascending=False)

print(f"Total months in dataset: {total_months}")
print("\nTop 20 most loyal items:")
print(loyalty.head(20).to_string(index=False))

# Plot top 20
top_loyal = loyalty.head(20)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_loyal['name'], top_loyal['loyalty_score'], color='#0066cc')
ax.set_title('Item Loyalty Score (% of months purchased)', fontsize=14)
ax.set_xlabel('% of months you bought this item')
ax.axvline(50, color='red', linestyle='--', linewidth=1, label='50% threshold')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()